<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


### Execution notes
Run the cells in order. Requests use the exact Wikipedia revision specified in the lab, with a permanent-link alternative and an unchanged local HTML snapshot in `data/`. Parsing removes citations, preserves linked text, and represents an absent payload mass as missing rather than zero. The numeric flight-number rule follows the provided lab snippet. The final cell exports `spacex_web_scraped.csv`.


First let's import required packages for this lab


In [1]:
# Install only missing dependencies into the current notebook's Python.
import importlib.util
import subprocess
import sys

packages = {"bs4": "beautifulsoup4", "requests": "requests", "pandas": "pandas"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [2]:
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
from copy import copy
from pathlib import Path

DATA_DIR = Path("capstone/data") if Path("capstone").is_dir() else Path("data")

/Users/vavanfarizzaman/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/var/folders/kf/vykd4ky17jq2467hdrt_0hsm0000gn/T/ipykernel_3729/373490998.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def clean_text(cell):
    """Preserve linked text while removing footnotes without changing the soup."""
    fragment = copy(cell)
    for reference in fragment.select("sup, .mw-editsection"):
        reference.decompose()
    text = unicodedata.normalize("NFKC", fragment.get_text(" ", strip=True))
    return re.sub(r"\s+", " ", text).strip()


def date_time(cell):
    text = clean_text(cell)
    match = re.search(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", text)
    if match is None:
        return [text.strip(" ,"), None]
    return [text[:match.start()].strip(" ,"), match.group()]


def booster_version(cell):
    return clean_text(cell)


def landing_status(cell):
    return clean_text(cell)


def get_mass(cell):
    text = clean_text(cell)
    match = re.search(r"[\d,]+(?:\.\d+)?\s*kg\b", text)
    # Preserve classifications such as "Classified"; a blank is unknown, not zero.
    return match.group() if match else (text or None)


def extract_column_from_header(cell):
    name = clean_text(cell)
    name = re.sub(r"\(\s+", "(", name)
    name = re.sub(r"\s+\)", ")", name)
    return name if name and not name.isdigit() else None

To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# Both URLs identify the exact revision requested by the lab.
snapshot_urls = [
    static_url,
    "https://en.wikipedia.org/wiki/Special:PermanentLink/1027686922",
]
response = None
for url in snapshot_urls:
    try:
        candidate = requests.get(url, headers=headers, timeout=30)
        candidate.raise_for_status()
        response = candidate
        html = response.text
        print(f"Retrieved Wikipedia revision 1027686922: {url}")
        break
    except requests.RequestException:
        continue
else:
    snapshot = DATA_DIR / "falcon9_wikipedia_1027686922.html"
    if not snapshot.is_file():
        raise RuntimeError("Wikipedia requests failed and the local revision snapshot is missing.")
    html = snapshot.read_text()
    print(f"Network unavailable; using the saved Wikipedia revision: {snapshot}")

Retrieved Wikipedia revision 1027686922: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(html, "html.parser")

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all("table")

Identify launch tables by their headers, so extra layout tables do not change which data is selected.

In [9]:
launch_tables = [
    table for table in html_tables
    if table.find("tr") is not None
    and "Flight No." in [
        extract_column_from_header(th)
        for th in table.find("tr").find_all("th")
    ]
]
if not launch_tables:
    raise ValueError("The page contains no launch tables with a Flight No. header.")
first_launch_table = launch_tables[0]
print(str(first_launch_table)[:2000])

<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
<tbody id="mwmg"><tr id="mwmw"><th id="mwnA" scope="col">Flight No.</th>
<th id="mwnQ" scope="col">Date and<br id="mwng"/>time (<a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">UTC</a>)</th>
<th id="mwoA" scope="col"><a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">Version,<br id="mwog"/>Booster</a> <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boosters are designated with a construction serial number and an optional flight number when reused, e.g. B1021.1 and B1021.

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
# Only the header row contains column names; other <th> cells are flight numbers.
for th in first_launch_table.find("tr").find_all("th"):
    name = extract_column_from_header(th)
    if name:
        column_names.append(name)

Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time (UTC)', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
# Preserve the lab's output column names and split the timestamp into Date/Time.
output_columns = []
for name in column_names:
    if name.startswith("Date and time"):
        continue
    output_columns.append("Version Booster" if name == "Version, Booster" else name)
output_columns.extend(["Date", "Time"])
launch_dict = {name: [] for name in output_columns}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
# Reset the lists so rerunning this extraction cell cannot duplicate launches.
launch_dict = {name: [] for name in output_columns}
extracted_row = 0
for table in launch_tables:
    for row in table.find_all("tr"):
        heading = row.find("th", recursive=False)
        flight_number = clean_text(heading) if heading else ""
        # Follow the lab's numeric-flight rule; skip commentary and future launches.
        if not flight_number.isdigit():
            continue
        cells = row.find_all("td", recursive=False)
        if len(cells) != 9:
            raise ValueError(f"Flight {flight_number}: expected 9 data cells, got {len(cells)}")
        date, time = date_time(cells[0])
        record = {
            "Flight No.": int(flight_number),
            "Date": date,
            "Time": time,
            "Version Booster": booster_version(cells[1]),
            "Launch site": clean_text(cells[2]),
            "Payload": clean_text(cells[3]),
            "Payload mass": get_mass(cells[4]),
            "Orbit": clean_text(cells[5]),
            "Customer": clean_text(cells[6]),
            "Launch outcome": clean_text(cells[7]),
            "Booster landing": landing_status(cells[8]),
        }
        for key in launch_dict:
            launch_dict[key].append(record[key])
        extracted_row += 1
print(f"Extracted {extracted_row} launch records.")

Extracted 121 launch records.


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
# Construct directly: unequal column lengths should raise, not be silently padded.
df = pd.DataFrame(launch_dict)
assert not df.empty
assert df["Flight No."].is_unique
assert df["Date"].ne("").all()
df.head()

,Flight No.,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Date,Time
0,1,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,None,LEO,SpaceX,Success,Failure (parachute),4 June 2010,18:45
1,2,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),None,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute),8 December 2010,15:43
2,3,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success,No attempt,22 May 2012,07:44
3,4,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt,8 October 2012,00:35
4,5,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt,1 March 2013,15:10


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [15]:
df.to_csv('spacex_web_scraped.csv', index=False)

## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
